In [ ]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d shreyanshpatel1/130k-real-vs-fake-face

Dataset URL: https://www.kaggle.com/datasets/shreyanshpatel1/130k-real-vs-fake-face
License(s): MIT
100% 12.2G/12.2G [02:13<00:00, 98.5MB/s]



In [ ]:
! unzip 130k-real-vs-fake-face.zip

Streaming output truncated to the last 5000 lines.
  inflating: images/real/65000.jpg   
  inflating: images/real/65001.jpg   
  inflating: images/real/65002.jpg   
  inflating: images/real/65003.jpg   
  inflating: images/real/65004.jpg   
  inflating: images/real/65005.jpg   
  inflating: images/real/65006.jpg   
  inflating: images/real/65007.jpg   
  inflating: images/real/65008.jpg   
  inflating: images/real/65009.jpg   
  inflating: images/real/65010.jpg   
  inflating: images/real/65011.jpg   
  inflating: images/real/65012.jpg   
  inflating: images/real/65013.jpg   
  inflating: images/real/65014.jpg   
  inflating: images/real/65015.jpg   
  inflating: images/real/65016.jpg   
  inflating: images/real/65017.jpg   
  inflating: images/real/65018.jpg   
  inflating: images/real/65019.jpg   
  inflating: images/real/65020.jpg   
  inflating: images/real/65021.jpg   
  inflating: images/real/65022.jpg   
  inflating: images/real/65023.jpg   
  inflating: images/real/65024.jpg   

In [13]:
import tensorflow as tf
from tensorflow import keras
from keras.applications.vgg16 import VGG16
from keras import Sequential
from keras.layers import Dropout,MaxPooling2D,Conv2D,Dense,BatchNormalization,Flatten


In [14]:
dataset_dir = "images"

train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(224,224),
    batch_size=16
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(224,224),
    batch_size=16
)

Found 133569 files belonging to 2 classes.
Using 106856 files for training.
Found 133569 files belonging to 2 classes.
Using 26713 files for validation.


In [15]:
def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

train_ds = train_ds.map(preprocess)
val_ds = val_ds.map(preprocess)

In [16]:
from tensorflow.keras.applications import MobileNetV2

conv_base = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

conv_base.trainable = False

# for layer in conv_base.layers[:-40]:
#     layer.trainable = False

In [17]:
# data augmentation and early stopping function

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import layers

data_aug = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)


In [18]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D

model = Sequential([
    tf.keras.layers.Input(shape=(224,224,3)),
    data_aug,
    conv_base,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dropout(0.4),
    Dense(1, activation='sigmoid')
])

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential_2 (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [19]:

from tensorflow.keras.optimizers import Adam
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [20]:
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3
)

Epoch 1/3
6679/6679 ━━━━━━━━━━━━━━━━━━━━ 577s 85ms/step - accuracy: 0.9314 - loss: 0.1789 - val_accuracy: 0.9747 - val_loss: 0.0758
Epoch 2/3
6679/6679 ━━━━━━━━━━━━━━━━━━━━ 549s 82ms/step - accuracy: 0.9739 - loss: 0.0762 - val_accuracy: 0.9823 - val_loss: 0.0518
Epoch 3/3
6679/6679 ━━━━━━━━━━━━━━━━━━━━ 534s 80ms/step - accuracy: 0.9782 - loss: 0.0620 - val_accuracy: 0.9847 - val_loss: 0.0434


In [21]:
model.save("deepfake_detector.keras")